# Project Test Console

Bu notebook genel amaçlıdır; başlangıç ayarları şu anda **Tessera** deposunu derlemek ve portable testlerini çalıştırmak için hazırlanmıştır.

## Kullanım
1. **Open in Colab** ile aç. Private repository kullanıyorsan Colab → **Secrets** bölümünde `GH_TOKEN` oluştur ve notebook erişimini etkinleştir. Public repository için token gerekmez.
2. Panelde repository, branch, kurulum komutları, test komutları ve zaman aşımı ayarlarını değiştirebilirsin. Tessera varsayılanlarını olduğu gibi bırakabilirsin.
3. **Testleri çalıştır** düğmesine bas.
4. Sonuç çıktığında **Raporu kopyala** düğmesine bas ve raporu doğrudan ChatGPT sohbetine yapıştır.

## Test komutu biçimi
Her satır bir testtir: `Test adı :: komut`. İsteğe bağlı araç kontrolü için sona `:: needs=clang++` eklenebilir. Eksik araç varsa test hata sayılmaz, **SKIP** olur. Kurulum komutları her satırda bir komut olacak şekilde girilir.

## Güvenlik ve kapsam
- Token yalnızca klonlama sırasında geçici ortam değişkeninde tutulur; test süreçlerine aktarılmaz ve rapora eklenmez.
- Komutlar Colab çalışma alanında çalıştırılır. Yalnızca güvendiğin repository ve komutları kullan.
- Bu notebook, yazdığın komutları Colab Linux ortamında çalıştırır. Gerçek Windows/MSVC/Detours/inline-hook/unwind/loader doğrulaması yerine geçmez.
- `Clone depth = 0` tam geçmişi indirir; `1` hızlı, sığ klonlamadır.


In [ ]:
#@title Tessera / General Project Validation Console { display-mode: "form" }
#@markdown Ayarlanabilir test paneli. Varsayılan profil Tessera içindir; tüm alanları başka projeler için değiştirebilirsin.

import os, re, sys, json, time, shutil, tempfile, subprocess, platform, datetime, pathlib, urllib.parse
import ipywidgets as W
from IPython.display import display, HTML, Javascript, clear_output
from google.colab import userdata

# ---------- UI ----------
wide = W.Layout(width="100%")
half = W.Layout(width="49%")
style = {"description_width": "150px"}

repo = W.Text(value="https://github.com/Jue-dev/Tessera.git", description="Git repository", placeholder="https://github.com/owner/repo.git", layout=wide, style=style)
branch = W.Text(value="issue-6/unwind-lifecycle", description="Branch / ref", layout=wide, style=style)
secret_name = W.Text(value="GH_TOKEN", description="Colab Secret adı", layout=half, style=style)
git_username = W.Text(value="x-access-token", description="Git kullanıcı adı", layout=half, style=style)
depth = W.BoundedIntText(value=1, min=0, max=100000, description="Clone depth (0=full)", layout=half, style=style)
timeout = W.BoundedIntText(value=600, min=10, max=7200, description="Test timeout (sn)", layout=half, style=style)
clone_timeout = W.BoundedIntText(value=180, min=10, max=1800, description="Clone timeout (sn)", layout=half, style=style)
install_clang = W.Checkbox(value=False, description="Clang yoksa kur (apt)", indent=False, layout=half)
keep_workspace = W.Checkbox(value=True, description="Çalışma alanını koru", indent=False, layout=half)
continue_after_failure = W.Checkbox(value=True, description="Bir test hata verse de devam et", indent=False, layout=half)

setup_commands = W.Textarea(
    value="",
    description="Kurulum komutları",
    placeholder="İsteğe bağlı — her satıra bir komut\nÖrn: cmake --version",
    rows=2, layout=wide, style=style
)
test_commands = W.Textarea(
    value=(
        "GCC · strict mock :: bash verify/windows_mock/run_windows_mock_tests.sh\n"
        "Clang · strict mock :: CXX=clang++ bash verify/windows_mock/run_windows_mock_tests.sh :: needs=clang++\n"
        "GCC · ASan + UBSan :: EXTRA_CXXFLAGS='-fsanitize=address,undefined -fno-omit-frame-pointer' ASAN_OPTIONS='detect_leaks=1:halt_on_error=1' UBSAN_OPTIONS='halt_on_error=1:print_stacktrace=1' bash verify/windows_mock/run_windows_mock_tests.sh"
    ),
    description="Test komutları",
    placeholder="Her satır: Test adı :: çalıştırılacak komut :: needs=clang++ (isteğe bağlı)",
    rows=6, layout=wide, style=style
)

run_button = W.Button(description="Testleri çalıştır", icon="play", button_style="success", layout=W.Layout(width="220px", height="42px"))
copy_button = W.Button(description="Raporu kopyala", icon="copy", button_style="primary", disabled=True, layout=W.Layout(width="180px", height="42px"))
status = W.HTML(value="<span style='color:#64748b'>Hazır. Ayarları kontrol et ve testleri başlat.</span>")
progress = W.Output(layout=W.Layout(width="100%", border="1px solid #d1d5db", padding="8px"))
report_box = W.Textarea(value="", description="Gönderilecek rapor", placeholder="Test bittikten sonra rapor burada oluşturulacak. Kopyala düğmesini kullan veya bu alandan tamamını seç.", rows=18, layout=wide, style=style)

def parse_test_lines(raw):
    parsed, errors = [], []
    for line_no, raw_line in enumerate(raw.splitlines(), 1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [part.strip() for part in line.split("::", 2)]
        if len(parts) < 2 or not parts[0] or not parts[1]:
            errors.append(f"Test komutu satır {line_no}: 'Test adı :: komut' biçimi gerekli.")
            continue
        needs = []
        if len(parts) == 3 and parts[2]:
            if not parts[2].startswith("needs="):
                errors.append(f"Test komutu satır {line_no}: üçüncü alan 'needs=program' biçiminde olmalı.")
                continue
            needs = [x.strip() for x in parts[2][6:].split(",") if x.strip()]
        parsed.append({"name": parts[0], "command": parts[1], "needs": needs})
    return parsed, errors

def safe_repo_label(value):
    try:
        p = urllib.parse.urlsplit(value)
        if p.scheme and p.netloc:
            return urllib.parse.urlunsplit((p.scheme, p.netloc, p.path, "", ""))
    except Exception:
        pass
    return value.split("?", 1)[0]

def redact(value, secret):
    value = str(value or "")
    if secret:
        value = value.replace(secret, "[REDACTED]")
    return value

def run_command(command, cwd, env, limit):
    started = time.time()
    try:
        proc = subprocess.run(
            command, cwd=str(cwd), env=env, shell=True, executable="/bin/bash",
            capture_output=True, text=True, timeout=limit
        )
        output = (proc.stdout or "") + (("\n[STDERR]\n" + proc.stderr) if proc.stderr else "")
        return proc.returncode, round(time.time() - started, 2), output
    except subprocess.TimeoutExpired as exc:
        out = (exc.stdout or "")
        err = (exc.stderr or "")
        if isinstance(out, bytes): out = out.decode("utf-8", "replace")
        if isinstance(err, bytes): err = err.decode("utf-8", "replace")
        return 124, round(time.time() - started, 2), f"TIMEOUT after {limit}s\n{out}\n[STDERR]\n{err}"
    except Exception as exc:
        return -1, round(time.time() - started, 2), f"{type(exc).__name__}: {exc}"

def capped(text, limit=18000):
    text = str(text or "")
    if len(text) <= limit:
        return text
    return text[:5000] + "\n\n... OUTPUT TRUNCATED ...\n\n" + text[-(limit - 5000):]

def on_copy(_):
    payload = json.dumps(report_box.value, ensure_ascii=False)
    js = """
(async () => {
  const text = PAYLOAD;
  try {
    await navigator.clipboard.writeText(text);
    alert("Tessera / Project Test Report panoya kopyalandı. Sohbete yapıştırabilirsin.");
  } catch (e) {
    const box = document.createElement("textarea");
    box.value = text;
    box.style.position = "fixed";
    box.style.opacity = "0";
    document.body.appendChild(box);
    box.focus(); box.select();
    const ok = document.execCommand("copy");
    box.remove();
    alert(ok ? "Rapor panoya kopyalandı." : "Otomatik kopyalama engellendi. Rapor alanına tıklayıp Ctrl+A, Ctrl+C kullan.");
  }
})();
""".replace("PAYLOAD", payload)
    display(Javascript(js))

copy_button.on_click(on_copy)

def on_run(_):
    run_button.disabled = True
    copy_button.disabled = True
    report_box.value = ""
    status.value = "<b style='color:#2563eb'>Çalışıyor…</b>"
    with progress:
        clear_output(wait=True)
        print("Hazırlık başlıyor…", flush=True)

    issues, results, setup_results = [], [], []
    report_lines = []
    started_all = time.time()
    workdir = pathlib.Path("/content/project-validation-workspace")
    secret = None
    commit = "NOT_AVAILABLE"
    clone_ok = False
    repo_url = repo.value.strip()
    branch_name = branch.value.strip()
    repo_display = safe_repo_label(repo_url)

    try:
        if not repo_url:
            raise ValueError("Git repository alanı boş.")
        if not (repo_url.startswith("https://") or repo_url.startswith("http://") or repo_url.startswith("git@")):
            raise ValueError("Repository adresi https://, http:// veya git@ ile başlamalı.")
        test_specs, parse_errors = parse_test_lines(test_commands.value)
        issues.extend({"stage": "Configuration", "detail": e} for e in parse_errors)
        if not test_specs:
            issues.append({"stage": "Configuration", "detail": "Çalıştırılacak geçerli test komutu bulunamadı."})

        try:
            if secret_name.value.strip():
                secret = userdata.get(secret_name.value.strip())
        except Exception:
            secret = None  # Public repositories do not require a token.

        if workdir.exists():
            shutil.rmtree(workdir)

        git_env = os.environ.copy()
        git_env.update({"GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "Never"})
        askpass_dir = None
        if secret:
            askpass_dir = pathlib.Path(tempfile.mkdtemp(prefix="project_test_askpass_"))
            askpass_file = askpass_dir / "askpass.sh"
            askpass_file.write_text(
                '#!/bin/sh\n'
                'case "$1" in\n'
                '  *Username*) printf "%s\\n" "$PROJECT_GIT_USERNAME" ;;\n'
                '  *) printf "%s\\n" "$PROJECT_GIT_TOKEN" ;;\n'
                'esac\n',
                encoding="utf-8"
            )
            askpass_file.chmod(0o700)
            git_env.update({
                "GIT_ASKPASS": str(askpass_file),
                "PROJECT_GIT_USERNAME": git_username.value.strip() or "x-access-token",
                "PROJECT_GIT_TOKEN": secret
            })

        clone_cmd = ["git", "clone"]
        if depth.value > 0:
            clone_cmd += ["--depth", str(depth.value)]
        if branch_name:
            clone_cmd += ["--single-branch", "--branch", branch_name]
        clone_cmd += [repo_url, str(workdir)]

        try:
            with progress:
                print(f"Repository alınıyor: {repo_display} [{branch_name or 'default branch'}]", flush=True)
            clone = subprocess.run(
                clone_cmd, env=git_env, capture_output=True, text=True,
                timeout=clone_timeout.value
            )
            clone_out = redact((clone.stdout or "") + (clone.stderr or ""), secret)
            if clone.returncode != 0:
                issues.append({"stage": "Git clone", "detail": capped(clone_out or f"git clone exit={clone.returncode}")})
            else:
                clone_ok = True
                commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=workdir, text=True).strip()
                with progress:
                    print(f"Repository hazır ✓ | Commit: {commit[:12]}", flush=True)
        except subprocess.TimeoutExpired:
            issues.append({"stage": "Git clone", "detail": f"Clone işlemi {clone_timeout.value} saniyede zaman aşımına uğradı."})
        except Exception as exc:
            issues.append({"stage": "Git clone", "detail": f"{type(exc).__name__}: {exc}"})
        finally:
            if askpass_dir and askpass_dir.exists():
                shutil.rmtree(askpass_dir, ignore_errors=True)
            git_env.pop("PROJECT_GIT_TOKEN", None)
            git_env.pop("PROJECT_GIT_USERNAME", None)
            secret = None

        if clone_ok:
            env = os.environ.copy()
            # Setup steps are deliberately user-configurable and run inside the cloned project.
            for line_no, command in enumerate(setup_commands.value.splitlines(), 1):
                command = command.strip()
                if not command or command.startswith("#"):
                    continue
                with progress:
                    print(f"[SETUP {line_no}] {command}", flush=True)
                code, duration, output = run_command(command, workdir, env, timeout.value)
                record = {"name": f"Setup {line_no}", "command": command, "exit": code, "duration": duration, "output": capped(output)}
                setup_results.append(record)
                if code != 0:
                    issues.append({"stage": record["name"], "detail": record["output"] or f"exit={code}"})
                    if not continue_after_failure.value:
                        break

            if install_clang.value and shutil.which("clang++") is None:
                with progress:
                    print("[TOOLS] Clang kuruluyor…", flush=True)
                code, duration, output = run_command("apt-get update -qq && apt-get install -y clang", workdir, env, 300)
                setup_results.append({"name": "Install Clang", "command": "apt-get update -qq && apt-get install -y clang", "exit": code, "duration": duration, "output": capped(output)})
                if code != 0:
                    issues.append({"stage": "Install Clang", "detail": capped(output)})

            halted = bool(setup_results and setup_results[-1]["exit"] != 0 and not continue_after_failure.value)
            for spec in test_specs:
                if halted:
                    results.append({"name": spec["name"], "status": "SKIP", "exit": None, "duration": 0, "command": spec["command"], "output": "Önceki hata nedeniyle devam ayarı kapalı."})
                    continue
                missing = [tool for tool in spec["needs"] if shutil.which(tool) is None]
                if missing:
                    results.append({"name": spec["name"], "status": "SKIP", "exit": None, "duration": 0, "command": spec["command"], "output": "Eksik araç: " + ", ".join(missing)})
                    with progress:
                        print(f"↷ {spec['name']} — SKIP (eksik: {', '.join(missing)})", flush=True)
                    continue

                with progress:
                    print(f"[TEST] {spec['name']}", flush=True)
                code, duration, output = run_command(spec["command"], workdir, env, timeout.value)
                status_value = "PASS" if code == 0 else "FAIL"
                rec = {"name": spec["name"], "status": status_value, "exit": code, "duration": duration, "command": spec["command"], "output": capped(output)}
                results.append(rec)
                with progress:
                    print(f"{'✓' if code == 0 else '✗'} {spec['name']} — {status_value} ({duration}s)", flush=True)
                if code != 0:
                    issues.append({"stage": spec["name"], "detail": rec["output"] or f"exit={code}"})
                    if not continue_after_failure.value:
                        halted = True

    except Exception as exc:
        issues.append({"stage": "Runner", "detail": f"{type(exc).__name__}: {exc}"})

    elapsed = round(time.time() - started_all, 2)
    passed = sum(x["status"] == "PASS" for x in results)
    failed = sum(x["status"] == "FAIL" for x in results)
    skipped = sum(x["status"] == "SKIP" for x in results)
    setup_failed = any(x["exit"] != 0 for x in setup_results)
    if issues or failed or setup_failed or not clone_ok:
        overall = "FAIL"
    elif skipped:
        overall = "PARTIAL"
    elif passed:
        overall = "PASS"
    else:
        overall = "FAIL"

    commit_url = ""
    match = re.search(r"github\.com[:/]([^/]+/[^/.]+?)(?:\.git)?$", repo_url)
    if match and commit != "NOT_AVAILABLE":
        commit_url = f"https://github.com/{match.group(1)}/commit/{commit}"

    report_lines = [
        "PROJECT VALIDATION REPORT",
        f"Overall: {overall}",
        f"Repository: {repo_display}",
        f"Branch: {branch_name or 'default'}",
        f"Commit: {commit}",
        f"Commit URL: {commit_url or 'not available'}",
        f"Timestamp UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}",
        f"Environment: {platform.platform()} | Python {platform.python_version()}",
        f"Duration: {elapsed}s",
        f"Results: {passed} PASS / {failed} FAIL / {skipped} SKIP",
        "",
        "SETUP RESULTS:"
    ]
    if not setup_results:
        report_lines.append("(none)")
    for rec in setup_results:
        report_lines += [f"[{'PASS' if rec['exit'] == 0 else 'FAIL'}] {rec['name']} | exit={rec['exit']} | {rec['duration']}s", f"COMMAND: {rec['command']}", rec["output"], ""]

    report_lines.append("TEST RESULTS:")
    if not results:
        report_lines.append("(no tests executed)")
    for rec in results:
        report_lines += [
            f"[{rec['status']}] {rec['name']} | exit={rec['exit']} | {rec['duration']}s",
            f"COMMAND: {rec['command']}",
            rec["output"],
            ""
        ]

    report_lines.append("ISSUE POOL:")
    if not issues:
        report_lines.append("(empty)")
    for i, issue in enumerate(issues, 1):
        report_lines += [f"--- ISSUE {i}: {issue['stage']} ---", issue["detail"], "--- END ISSUE ---", ""]

    report_lines += ["", "END OF REPORT"]
    report_text = "\n".join(report_lines)
    report_box.value = report_text
    copy_button.disabled = False
    status_color = {"PASS": "#15803d", "PARTIAL": "#b45309", "FAIL": "#b91c1c"}[overall]
    status.value = (
        f"<div style='padding:10px;border-left:5px solid {status_color};background:#f8fafc'>"
        f"<b style='color:{status_color};font-size:18px'>{overall}</b> "
        f"— {passed} başarılı, {failed} hatalı, {skipped} atlandı. "
        f"<b>Rapor hazır:</b> 'Raporu kopyala' düğmesine basıp buraya yapıştır."
        f"</div>"
    )
    with progress:
        print(f"\nTamamlandı: {overall} | {elapsed}s", flush=True)
        print("Rapor hazır. 'Raporu kopyala' düğmesini kullan.", flush=True)

    if clone_ok and not keep_workspace.value and workdir.exists():
        shutil.rmtree(workdir, ignore_errors=True)

    run_button.disabled = False

run_button.on_click(on_run)

display(W.VBox([
    W.HTML("<h2 style='margin:0'>Project Test Console</h2><p style='color:#64748b;margin-top:4px'>Genel amaçlı test paneli · başlangıç ayarları Tessera için hazırlanmıştır · kod hücresi ayarları değiştirmek dışında gerekli değildir.</p>"),
    W.HTML("<b>1 · Repository ve erişim</b>"),
    repo, branch, W.HBox([secret_name, git_username]), W.HBox([depth, clone_timeout]),
    W.HTML("<b>2 · Derleme / test ayarları</b>"),
    setup_commands, test_commands, W.HBox([timeout, install_clang]),
    W.HBox([continue_after_failure, keep_workspace]),
    W.HBox([run_button, copy_button]),
    status,
    W.HTML("<b>Çalışma günlüğü</b>"),
    progress,
    W.HTML("<b>3 · Bana göndereceğin sonuç</b><p style='color:#64748b;margin-top:0'>Test tamamlandığında rapor burada görünür. Kopyala düğmesi panoya alır; çalışmazsa metin alanından Ctrl+A / Ctrl+C yap.</p>"),
    report_box
], layout=W.Layout(width="100%", gap="10px")))

print("Panel hazır. Ayarları düzenle ve 'Testleri çalıştır' düğmesine bas.")